# STEP02. 사람 라벨링

규칙 후보를 독립적으로 판정할 고정 표본을 준비하고 사람 판정 지침과 블라인드 시트를 제공한다.
이 단계에는 성능 평가·수요 분석·앱 분류체계 설계·LLM 실행을 포함하지 않는다.

입력은 STEP 00의 `data/processed/services_master.csv`이다.
집중 검수 목록은 호환성을 위해 기존 `outputs/01_eda/tables/goldset_candidates.csv`, `goldset_counts.csv` 경로에 저장하되 **생성 책임은 이 노트북에만** 둔다.
라벨링 설계·채점 키·차수별 블라인드 시트는 `outputs/02_labeling/tables/`에 저장한다.
사람이 작성한 답안은 `outputs/02_labeling/labels/<round>_<labeler>.csv`에 별도로 보관한다.

기존 표본·이중 판정·집중 검수 목록의 서비스 ID를 보존한다(건수는 아래 코드 셀 출력 참조). 시드·층화 규칙은 `src/labeling/sampling.py`를 따른다.

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Markdown
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)
print("ROOT =", ROOT.name, "| pandas", pd.__version__, "| python", sys.version.split()[0])
from src.labeling import sampling as SMP
OUT = ROOT / "outputs/02_labeling"; TBL = OUT / "tables"; LABELS = OUT / "labels"
TBL.mkdir(parents=True, exist_ok=True)
m = pd.read_csv(ROOT / "data/processed/services_master.csv", dtype=str, encoding="utf-8-sig", keep_default_na=False, na_values=[""])

for c in ["youth_tag", "youth_text"]:
    m[c] = m[c].map({"True": True, "False": False})

m["youth_age"] = m["youth_age"].map({"True": True, "False": False})
meta = json.loads((ROOT / "data/processed/services_master.meta.json").read_text(encoding="utf-8"))
assert m.servId.is_unique and len(m) == 5288


ROOT = repo | pandas 2.3.3 | python 3.13.18


## 집중 검수 목록 준비

기존 청년 프로그램·고립 언급 조건과 ID를 유지한다. 목록은 전체 서비스의 무작위 표본이 아니다.

In [2]:
LEGACY = ROOT / "outputs/01_eda/tables"
LEGACY.mkdir(parents=True, exist_ok=True)
pt = m.provision_type.fillna("")
has_prog = pt.str.contains(r"프로그램/서비스\(서비스\)|(?:^|, )기타(?:$|, )")
single_prog = pt.isin(["프로그램/서비스(서비스)", "기타"])
a = m.youth_tag & has_prog
b = m.isolation_explicit == "예"
GS = ["gs_A_summary", "gs_B_benefit", "gs_C_external", "gs_C_source", "gs_C_year", "gs_C_region", "gs_D_target_group", "gs_final_status"]
cols = ["servId", "source", "servNm", "org", "region_sido", "region_sgg", "provision_type", "life_tags", "elig_youth", "isolation_explicit",
        "target_group", "need_area_q21", "detail_url", "summary_text", "target_text", "benefit_text"]
gold = m.loc[a | b, cols].copy()
gold.insert(2, "in_a_youth_program", a[a | b].values); gold.insert(3, "in_b_isolation", b[a | b].values)

for c in GS: gold[c] = ""

existing_gold = LEGACY / "goldset_candidates.csv"

if existing_gold.exists():
    previous = pd.read_csv(existing_gold, dtype=str, encoding="utf-8-sig")
    assert set(previous.servId) == set(gold.servId), "집중 검수 ID 변경: 저장 중단"
    if previous[GS].fillna("").ne("").any().any():
        raise ValueError("집중 검수 판정값이 있어 덮어쓰지 않습니다")

gold.to_csv(LEGACY / "goldset_candidates.csv", index=False, encoding="utf-8-sig")
gold_counts = pd.DataFrame([
    {"set": "(a) 청년 태그 & 제공유형에 프로그램/서비스 또는 기타 포함", "n": int(a.sum())},
    {"set": "(a') 위 조건에서 제공유형이 단일(프로그램/서비스 또는 기타)", "n": int((m.youth_tag & single_prog).sum())},
    {"set": "(b) 본문 고립·은둔 언급", "n": int(b.sum())},
    {"set": "(a)∩(b)", "n": int((a & b).sum())},
    {"set": "합집합 = 저장 행 수", "n": len(gold)}])
gold_counts.to_csv(LEGACY / "goldset_counts.csv", index=False, encoding="utf-8-sig")
display(gold_counts)


,set,n
0,(a) 청년 태그 & 제공유형에 프로그램/서비스 또는 기타 포함,280
1,(a') 위 조건에서 제공유형이 단일(프로그램/서비스 또는 기타),224
2,(b) 본문 고립·은둔 언급,19
3,(a)∩(b),4
4,합집합 = 저장 행 수,295


# PART A. 왜 별도 표본이 필요한가

정답셋 후보는 청년 태그가 있는 프로그램형 서비스와 고립·은둔 언급 서비스만 모은 **집중 검수 목록**이다. 규칙이 영역을 붙이지 못한 서비스나 청년 태그가 없는 서비스는 거의 들어 있지 않아 **누락을 평가할 수 없다.** 그래서 전체 서비스에서 따로 표본을 뽑는다.

In [3]:
youth_any = m.youth_tag | m.youth_text | m.youth_age.eq(True)
area_hit = m.need_area_q21.ne("N")
g = m.servId.isin(gold.servId)
cover = pd.DataFrame([
    {"구분": "전체", "n": len(m), "청년 근거 없음": int((~youth_any).sum()), "영역 후보 없음(N)": int((~area_hit).sum())},
    {"구분": "정답셋 후보", "n": int(g.sum()), "청년 근거 없음": int((g & ~youth_any).sum()), "영역 후보 없음(N)": int((g & ~area_hit).sum())},
])
display(cover)
display(Markdown(f"**관찰:** 영역 후보가 없는 서비스는 전체 {int((~area_hit).sum()):,}건인데 정답셋 후보에는 {int((g & ~area_hit).sum()):,}건, "
                 f"청년 근거가 없는 서비스는 전체 {int((~youth_any).sum()):,}건인데 정답셋 후보에는 {int((g & ~youth_any).sum()):,}건이다."))


,구분,n,청년 근거 없음,영역 후보 없음(N)
0,전체,5288,3694,2230
1,정답셋 후보,295,10,159


**관찰:** 영역 후보가 없는 서비스는 전체 2,230건인데 정답셋 후보에는 159건, 청년 근거가 없는 서비스는 전체 3,694건인데 정답셋 후보에는 10건이다.

# PART B. 표본 설계

- **S Q1 층화 무작위:** 출처 × 청년 근거(태그·원문·연령 중 하나라도) × 영역 후보 유무 = 8개 층, 층마다 같은 수. 가중치 = N_h / n_h. 정밀도와 **재현율(누락)**을 함께 추정한다.
- **P 영역별 정밀도 보충:** 영역 후보가 있는 서비스에서 영역마다 최대 k건. 후보가 드문 영역의 정밀도를 보기 위한 비가중 표본이다.
- **J job_relevance 원인 구분:** `job_relevance_signal` 층(규칙 누락 후보 / 실제 무관 후보 / 원문 부족 후보)과 규칙 값 1·2 층.
- 20%는 두 명이 따로 라벨링해 일치도를 본다. seed와 층별 표본 수는 `src/labeling/sampling.py` 상단 상수다.

In [4]:
existing_key = TBL / "labeling_key.csv"
KEY_FROZEN = existing_key.exists()

if KEY_FROZEN:
    # 표본은 설계 시점(2026-11-02) 마스터의 층으로 고정한다. 이후 규칙이 바뀌어도(v3 등) 다시 뽑지 않는다.
    # 다시 뽑으면 층 구성이 달라져 기존 ID 검사에서 멈추고, 층 가중치(N_h/n_h)의 기준 시점도 바뀐다.
    key = pd.read_csv(existing_key, encoding="utf-8-sig", keep_default_na=False, na_values=[""],
                      dtype={"servId": str, "s_stratum": str, "p_area": str, "j_stratum": str})
    for c in ["in_S", "in_P", "in_J", "in_goldset_candidates", "double_label"]:
        key[c] = key[c].astype(str).eq("True")
    design_summary = pd.read_csv(TBL / "labeling_design_summary.csv", encoding="utf-8-sig")
else:
    design_summary, key = SMP.build_design(m, set(gold.servId))

display(design_summary.round(2))
r1 = key.in_S | key.in_P; r2 = key.in_J
design_counts = pd.DataFrame([
    {"구분": "합집합 (서비스 수)", "n": len(key)},
    {"구분": "S Q1 층화 무작위", "n": int(key.in_S.sum())},
    {"구분": "P 영역별 정밀도 보충", "n": int(key.in_P.sum())},
    {"구분": "J job_relevance 원인 구분", "n": int(key.in_J.sum())},
    {"구분": "1차 시트 (S ∪ P)", "n": int(r1.sum())},
    {"구분": "2차 시트 (J)", "n": int(r2.sum())},
    {"구분": "1·2차 모두 포함", "n": int((r1 & r2).sum())},
    {"구분": "이중 라벨 대상", "n": int(key.double_label.sum())},
    {"구분": "정답셋 후보와 겹침", "n": int(key.in_goldset_candidates.sum())},
])
display(design_counts)
display(Markdown(f"**관찰:** S는 층마다 최대 {SMP.N_PER_STRATUM_S}건, P는 영역마다 최대 {SMP.K_PER_AREA_P}건, "
                 f"J는 층별 {SMP.N_PER_STRATUM_J}. 1차 시트 {int(r1.sum())}건, 2차 시트 {int(r2.sum())}건 "
                 f"(라벨링 서비스 합계 {len(key)}건, 이중 라벨 {int(key.double_label.sum())}건)."))


,design,stratum,N_h,n_h,weight
0,S Q1 층화 무작위,central|noyouth|areahit,221,20,11.05
1,S Q1 층화 무작위,central|noyouth|noarea,63,20,3.15
2,S Q1 층화 무작위,central|youth|areahit,153,20,7.65
3,S Q1 층화 무작위,central|youth|noarea,24,20,1.20
4,S Q1 층화 무작위,local|noyouth|areahit,2575,20,128.75
5,S Q1 층화 무작위,local|noyouth|noarea,879,20,43.95
6,S Q1 층화 무작위,local|youth|areahit,1128,20,56.40
7,S Q1 층화 무작위,local|youth|noarea,245,20,12.25
8,P 영역별 정밀도 보충,q21_01,7,7,1.00
9,P 영역별 정밀도 보충,q21_02,10,8,1.25


,구분,n
0,합집합 (서비스 수),352
1,S Q1 층화 무작위,160
2,P 영역별 정밀도 보충,79
3,J job_relevance 원인 구분,120
4,1차 시트 (S ∪ P),236
5,2차 시트 (J),120
6,1·2차 모두 포함,4
7,이중 라벨 대상,74
8,정답셋 후보와 겹침,53


**관찰:** S는 층마다 최대 20건, P는 영역마다 최대 8건, J는 층별 {'keyword_present': 30, 'text_no_keyword': 30, 'short_text': 20, 'rule_2': 20, 'rule_1': 20}. 1차 시트 236건, 2차 시트 120건 (라벨링 서비스 합계 352건, 이중 라벨 74건).

# PART C. 라벨링 시트

라벨러에게는 차수별 **blind 시트**만 준다. 규칙 후보 값은 시트에 없고 `labeling_key.csv`에만 있다. 1차(Q1 영역)를 먼저 끝내면 Q1 분석의 공급 쪽 숫자를 검증할 수 있고, 2차(job_relevance)는 추천 요소용이라 Q1과 독립이며 **선택(이번 범위 아님)**이다. 시트는 지우지 않고 보관한다.

In [5]:
sheets = {r: SMP.make_sheet(m, key, r) for r in ["round1_q1", "round2_job"]}
if not KEY_FROZEN:
    design_summary.to_csv(TBL / "labeling_design_summary.csv", index=False, encoding="utf-8-sig")
    key.to_csv(TBL / "labeling_key.csv", index=False, encoding="utf-8-sig")

for r, sh in sheets.items():
    path = TBL / f"labeling_sheet_{r}.csv"
    if path.exists():
        previous = pd.read_csv(path, dtype=str, encoding="utf-8-sig", keep_default_na=False)
        assert list(previous.servId) == list(sh.servId), "블라인드 시트 ID·순서 변경"
        human_cols = [col for col in previous if col.startswith("h_") or col in ["labeler", "labeled_at"]]
        if previous[human_cols].ne("").any().any():
            raise ValueError(f"작성된 판정값이 있어 덮어쓰지 않습니다: {path}")
    sh.to_csv(TBL / f"labeling_sheet_{r}.csv", index=False, encoding="utf-8-sig")

display(pd.DataFrame([{"sheet": f"labeling_sheet_{r}.csv", "rows": len(sh), "double_label": int(sh.double_label.sum()),
                       "label_columns": ", ".join(c for c in sh.columns if c.startswith("h_"))} for r, sh in sheets.items()]))


,sheet,rows,double_label,label_columns
0,labeling_sheet_round1_q1.csv,236,53,"h_q21_01, h_q21_02, h_q21_03, h_q21_04, h_q21_05, h_q21_06, h_q21_07, h_q21_08, h_q21_09, h_q21_10, h_q21_11, h_q21_..."
1,labeling_sheet_round2_job.csv,120,21,"h_job_relevance, h_job_reason, h_job_evidence, h_note"


## PART C-2. 보충 표본: 청년 범위 youth_scope (2026-11-04)

**목적:** 기존 S 층은 '청년 근거 유무'로만 나뉘어 있어 U(근거 없음)·C(연령 제외)와 B의 근거 종류별 정확도를 따로 추정하기 어렵다. 기존 표본은 그대로 두고 보충 표본 R을 따로 뽑는다(`src/labeling/supplement.py`).

- 층: `youth_scope × source` 8개, 층마다 최대 15건, 가중치 = N_h / n_h. 모집단에서 기존 표본 352건과 규칙 개발 사례(경계 사례 검토 + `review_overrides.TO_N`)를 뺐다.
- 이중 판정 약 20%. blind 시트에는 규칙 값이 없다. 판정 칸: `h_youth_scope`(A/B/U/C), `h_youth_scope_evidence`, `h_target_condition`.
- 기존 채점 키 중 규칙 개발 사례와 겹치는 서비스는 `labeling_key_rule_dev_overlap.csv`에 표시한다. 독립 평가 지표는 이 서비스를 뺀 값과 포함한 값을 함께 낸다.
- **완료 기준 분리:** 시트 준비 완료 ≠ 라벨 정확도 검증. 사람 판정이 들어오기 전에는 정확도 지표를 계산하지 않는다.

In [6]:
from src.labeling import supplement as SUP
sup = SUP.build(m.astype({"youth_scope_review": str}), key.astype({c: str for c in ["in_S", "in_P", "in_J"]}))
for name, t in sup.items():
    path = TBL / f"{name}.csv"
    if name.startswith("labeling_sheet") and path.exists():
        prev = pd.read_csv(path, dtype=str, encoding="utf-8-sig", keep_default_na=False)
        if prev[[c for c in prev if c.startswith("h_") or c in ["labeler", "labeled_at"]]].ne("").any().any():
            continue                                   # 작성된 판정값이 있으면 덮어쓰지 않는다
    t.to_csv(path, index=False, encoding="utf-8-sig")
kr = sup["labeling_key_round1b_youth_scope"]
display(kr.groupby("r_stratum").agg(N_h=("r_N_h", "first"), n_h=("r_n_h", "first"), weight=("r_weight", "first")).reset_index())
display(sup["labeling_youth_scope_coverage"])
ov = sup["labeling_key_rule_dev_overlap"]
display(Markdown(f"**관찰:** 보충 표본 {len(kr)}건(이중 판정 {int(kr.double_label.astype(str).eq('True').sum())}건). "
                 f"기존 채점 키 {len(ov)}건 중 규칙 개발 사례와 겹치는 서비스 {int(ov.in_rule_dev.sum())}건."))

,r_stratum,N_h,n_h,weight
0,A|age_band,18,12,1.5
1,A|name_text,236,12,19.666666666666668
2,B|age_overlap,322,12,26.833333333333332
3,B|tag_only,721,12,60.083333333333336
4,B|target_text_mention,10,10,1.0
5,B|text_mention,7,7,1.0
6,C|age_excludes,257,12,21.416666666666668
7,U|age_may_refer_to_dependent,268,12,22.333333333333332
8,U|age_partial,403,12,33.583333333333336
9,U|conflict,41,12,3.4166666666666665


,youth_scope,youth_scope_basis,N_population,n_key,n_S,n_P,n_J
0,A,age_band,17,0.0,0.0,0.0,0.0
1,A,name_text,274,34.0,6.0,6.0,22.0
2,B,age_overlap,395,43.0,28.0,9.0,6.0
3,B,tag_only,829,78.0,43.0,16.0,23.0
4,B,target_text_mention,15,4.0,2.0,1.0,1.0
5,B,text_mention,8,0.0,0.0,0.0,0.0
6,C,age_excludes,285,16.0,6.0,4.0,6.0
7,U,age_may_refer_to_dependent,305,27.0,12.0,6.0,9.0
8,U,age_partial,428,14.0,6.0,5.0,3.0
9,U,conflict,56,5.0,2.0,1.0,2.0


**관찰:** 보충 표본 137건(이중 판정 24건). 기존 채점 키 352건 중 규칙 개발 사례와 겹치는 서비스 17건.

## PART C-3. 보충 표본: 규칙 간 불일치 (2026-11-05)

**목적:** `services_master`와 팀 라벨 파일(`data/raw/team_labels/youth_service_labels_v1.csv`, 1,415건)은 둘 다 사람 검수 전이다. 어느 쪽도 정답으로 보지 않고, 외출 판정이 갈리는 서비스를 사람이 판정해 두 규칙을 함께 평가한다.

- 원인: 외출필요도가 같으면 `agree`. 다르면 해석 차이(`interp|apply·use·both`)와 한쪽만 N(`one_side_N|누가|어느 쪽`)으로 나눈다. **한쪽만 N은 오분류가 아니라 판단 보류일 수 있다.**
- 표본: 해석 차이는 전수, 한쪽만 N은 층마다 12건(가중치 N_h/n_h). 1차 시트(S∪P)에 이미 있는 서비스는 그 판정을 쓰므로 뺐다. 규칙 개발 사례는 빼지 않고 `in_rule_dev`로 표시한다.
- 판정 칸: `h_apply_barrier`, `h_apply_visit`(방문 필수 / 방문 경로만 안내 / 비방문 경로 있음 / 신청 정보 없음), `h_apply_evidence`, `h_use_outing`, `h_use_outing_evidence`.

## PART C-4. 말뭉치 문장별 정답 양식

말뭉치 300건의 `정답_영역`·`정답_단계`·`정답_위기`·`기대행동`은 유형마다 30문장이 같은 값(유형 기본값)이다. 원자료는 그대로 두고, 키 파일에 `유형_기본*`으로 이름을 바꿔 두며, blind 시트(`corpus_gold_sheet.csv`)에 문장별 정답을 단다(`src/labeling/corpus_gold.py`).

In [7]:
dis = SUP.build_disagreement(m.astype({"youth_scope_review": str}), key.astype({c: str for c in ["in_S", "in_P", "in_J"]}))
for name, t in dis.items():
    path = TBL / f"{name}.csv"
    if name.startswith("labeling_sheet") and path.exists():
        prev = pd.read_csv(path, dtype=str, encoding="utf-8-sig", keep_default_na=False)
        if prev[[c for c in prev if c.startswith("h_") or c in ["labeler", "labeled_at"]]].ne("").any().any():
            continue
    t.to_csv(path, index=False, encoding="utf-8-sig")
display(dis["label_disagreement_summary"])
kc = dis["labeling_key_round1c_rule_disagreement"]
display(kc.groupby("c_stratum").agg(N_h=("c_N_h", "first"), n_h=("c_n_h", "first")).reset_index())
from src.labeling import corpus_gold as CG
corpus = CG.load()
display(CG.type_default_check(corpus))
ckey, csheet = CG.build(corpus)
CG.write_key(ckey, TBL / "corpus_gold_key.csv")
cpath = TBL / "corpus_gold_sheet.csv"
if not cpath.exists():
    csheet.to_csv(cpath, index=False, encoding="utf-8-sig")
CG.type_default_check(corpus).to_csv(TBL / "corpus_type_default_check.csv", index=False, encoding="utf-8-sig")
display(Markdown(f"**관찰:** 불일치 표본 {len(kc)}건(해석 차이 전수 {int(kc.c_stratum.str.startswith('interp').sum())}건, 이중 판정 {int(kc.double_label.astype(str).eq('True').sum())}건). "
                 f"말뭉치 판정 시트 {len(csheet)}건(이중 판정 {int(csheet.double_label.sum())}건). 사람 판정 전에는 정확도 지표를 계산하지 않는다."))

,cause,n
0,agree,1032
1,interp|apply,24
2,interp|both,1
3,interp|use,52
4,one_side_N|rule_N|apply,13
5,one_side_N|rule_N|use,49
6,one_side_N|team_N|apply,97
7,one_side_N|team_N|use,147


,c_stratum,N_h,n_h
0,interp|apply,23,23
1,interp|both,1,1
2,interp|use,45,45
3,one_side_N|rule_N|apply,13,12
4,one_side_N|rule_N|use,46,12
5,one_side_N|team_N|apply,82,12
6,one_side_N|team_N|use,145,12


,유형번호,정답_영역,정답_단계,정답_위기,기대행동,제약_표현
0,1,1,1,1,1,4
1,10,1,1,1,1,1
2,2,1,1,1,1,4
3,3,1,1,1,1,3
4,4,1,1,1,1,3
5,5,1,1,1,1,4
6,6,1,1,1,1,4
7,7,1,1,1,1,4
8,8,1,1,1,1,1
9,9,1,1,1,1,3


**관찰:** 불일치 표본 117건(해석 차이 전수 69건, 이중 판정 31건). 말뭉치 판정 시트 300건(이중 판정 74건). 사람 판정 전에는 정확도 지표를 계산하지 않는다.

# PART D. 라벨링 지침

**공통**
- 원문(서비스명·요약·지원대상·선정기준·지원내용·신청방법)만 보고 판단한다. 필요하면 `detail_url`을 열어 볼 수 있지만, 외부 자료로 판단했으면 `h_note`에 적는다.
- 판단 근거가 된 문장을 원문에서 **그대로 복사**해 근거 칸에 넣는다. 근거 문장이 없으면 값을 추정하지 않는다.
- 파일 이름: `outputs/02_labeling/labels/<round>_<labeler>.csv` (예: `round1_q1_jiwoo.csv`). `labeler`, `labeled_at` 칸을 채운다.
- `double_label=True`인 서비스는 두 명이 서로 보지 않고 각각 라벨링한다.

**1차 (Q1 영역) — `h_q21_01`~`h_q21_11`: 예 / 아니오 / 불명**
- '예': 지원 내용이나 목적이 그 영역의 지원을 **직접** 제공한다고 서술할 때. 여러 영역이 '예'일 수 있다.
- '아니오': 원문을 읽고 그 영역 지원이 아니라고 판단할 때.
- '불명': 원문이 너무 짧거나 모호해 판단할 수 없을 때.
- 지원 대상의 자격 조건에만 나오는 말(예: '주거급여 수급자')은 그 영역의 지원이 아니다.
- 필수 제외: 공동생활가정(그룹홈)은 ⑨가 아니다. 노숙인·청소년·피해자 쉼터는 ①이 아니다. 장애인 활동지원의 '일상생활 지원'은 ②가 아니다. 셰어하우스는 ⑪이다.
- `h_youth_target`: 신청자 본인 연령·대상이 19~34세 청년을 포함하면 예, 명확히 제외하면 아니오, 판단할 수 없으면 확인필요.

**1차 추가 — 신청 장벽 `h_apply_barrier` (0~3/N), 근거 `h_apply_evidence`**
- 정의: 신청서 제출부터 선정 결정까지 사용자가 직접 해야 하는 일.
- 0 온라인으로 신청 완결(또는 신청이 필요 없다고 **명시**) / 1 전화·우편·팩스·대리 신청 / 2 방문 신청 / 3 필수 면접 또는 여러 기관 방문.
- **가장 쉬운 경로 기준:** 방문과 온라인이 모두 가능하면 0이다. 다만 필수 면접이 있으면 경로와 관계없이 3이다.
- **'심사'는 3이 아니다.** 서류심사·심의는 기관이 하는 절차다.
- 실제 신청이 가능하다고 적힌 문장만 근거로 쓴다('홈페이지 참조', '온라인 신청 불가'는 온라인 근거가 아니다). '직접 지원'만 있으면 N.
- 예: "주민센터 또는 복지로에서 신청" → 0 / "본인 또는 대리인이 방문 신청" → 1 / "보건소 방문 접수" → 2 / "서류심사 후 개별면접으로 선정" → 3

**1차 추가 — 서비스 이용 외출 조건 `h_use_outing` (0~3/N), 근거 `h_use_outing_evidence`**
- 정의: 선정된 뒤 혜택을 받는 동안 해야 하는 외출(이용 조건으로 붙은 의무 포함).
- 0 집 안(계좌 입금, 배달·발송, 찾아오는 상담·돌봄) / 1 근처 짧은 외출(가맹점·이용권 사용) / 2 목적 있는 방문, 가끔 또는 단기(1회성 프로그램, 검진, 기관 상담) / 3 정기 외출(주 1회 이상 출석, 직업훈련·인턴·일경험처럼 기간이 정해진 과정).
- **대화 여부는 이용 외출 조건에 넣지 않는다.** 전화·화상 상담은 0이고, 상담이 대면인지는 판단 기준이 아니다.
- 이 값은 서비스가 요구하는 외출 조건이다. 청년의 외출 상태(F7)와 같은 척도가 아니므로 추천 자격을 판정하는 데 쓰지 않는다.

**2차 (job_relevance, 선택·이번 범위 아님) — `h_job_relevance`: 0 / 1 / 2 / N, `h_job_reason`**
- 2 직접(취업알선·일경험·직업훈련·구직수당), 1 간접(교육·자격·진로상담·자산형성), 0 무관(지원 내용이 직업과 무관함을 보여주는 문장이 있을 때), N 판단할 근거 없음.
- `h_job_reason`: 블라인드 판정자는 실제무관(0) / 정보부족(N)만 기록하고, 관련성 1·2는 근거 문장을 남긴다. 규칙누락·규칙정답은 블라인드 판정이 끝난 뒤 채점 키와 대조해 확정한다. 규칙을 추측해 쓰지 않는다.

In [8]:
from src.preprocess.keywords import Q21_LABELS
display(pd.DataFrame({"need_area": list(Q21_LABELS), "label": list(Q21_LABELS.values())}))


,need_area,label
0,q21_01,눈치 보지 않고 들러서 머물 수 있는 공간
1,q21_02,일상생활 회복 지원
2,q21_03,혼자 하는 취미·문화·체육활동 지원
3,q21_04,같이 하는 취미·문화·체육활동 지원
4,q21_05,전문가에 의한 상담·치료
5,q21_06,"또래, 멘토 등에 의한 상담·지지"
6,q21_07,교육·학습 지원
7,q21_08,취업(정보)·일 경험 지원
8,q21_09,공동생활(규칙적인 생활 등)
9,q21_10,경제적 지원


## 판정 자료 관리

1차(S∪P)는 영역·청년 대상, 2차(J)는 직업 관련성을 서로 독립적으로 판정한다. 이중 판정자는 서로의 답을 보지 않는다. 팀원 분담은 아직 확정되지 않았다.

채점 키와 규칙 후보는 라벨러에게 공유하지 않는다. 집중 검수 목록에는 규칙 값이 있으므로 블라인드 시트와 함께 배포하지 않는다. 표본과 집중 검수 사례는 평가용으로 보존하고 개발 프롬프트 예시로 사용하지 않는다.

라벨링 완료 전 정확도 수치는 생성하지 않는다. 성능 평가 함수(영역 정밀도·재현율, 일치도, 신청 장벽·이용 외출 조건의 정확도·quadratic weighted κ·N 비율 비교 `ordinal_agreement`)는 `src/labeling/evaluate.py`에 보존하며 이 노트북에서는 호출하지 않는다. Q1 탐색은 후보 라벨의 한계를 명시하면 라벨링 완료 전에도 가능하다.